# Import Modules

In [1]:
#high level modules
import os
import importlib.util
from functools import partial
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [9]:

def import_from_path(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

universal_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/functions/"
this_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/forecast_rollout/"

import_from_path("universals", os.path.join(universal_dir, "universal_functions.py"))
from universals import load_pickle_file

import_from_path("forecast_fx", os.path.join(this_dir, "forecast_functions.py"))
from forecast_fx import prep_features, static_regime_tiny, pulsed_regime_tiny, rollout_forecast

dump_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/DSS_Shiny/www/forecast_tiny/"


# Import models

In [3]:
model_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/tiny_model/dump/five_ten/"

model_1 = load_pickle_file("model_1.pkl", model_dir)
model_2 = load_pickle_file("model_2.pkl", model_dir)
model_3 = load_pickle_file("model_3.pkl", model_dir)
model_4 = load_pickle_file("model_4.pkl", model_dir)

# Import data

In [5]:
data_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/forecast_rollout/data/"

test = pd.read_csv(os.path.join(data_dir, "model_data.csv"))

test["date"] = pd.to_datetime(test["date"])

# need to remove additional variables
# reduce to fewer variables
drop_cols = ['total_solar_radiation_m2', 'mean_rel_hum_m2', 
       'pump_cfs_m2', 'pump_cfs_m3', 'nf_cfs_m2', 'nf_cfs_m3', 'nf_cfs_m4',
       'chipmunk_cfs_m2', 'chipmunk_cfs_m3', 'chipmunk_cfs_m4']

test = test.drop(columns = drop_cols)

First thing is that we need to prep the data for forecasting. This step uses a single date and grabs the data seven days in the future. Data are recoded as needed, and a forecast date column is created - this is the date that the forecast is made. 

In [6]:
# make a list of dates from 2024-05-01 to 2024-11-01

all_dates = pd.date_range(start="2024-05-01", end="2024-11-01", freq="D").strftime("%Y-%m-%d")

control_dataset = pd.concat([prep_features(one_date=d, data=test, regime="control") for d in all_dates])
pumping_dataset = pd.concat([prep_features(one_date=d, data=test, regime="altered") for d in all_dates])
control_dataset["forecast_date"] = pd.to_datetime(control_dataset["forecast_date"])
pumping_dataset["forecast_date"] = pd.to_datetime(pumping_dataset["forecast_date"])

# get the unique dates from the control/pumping datsets
unique_dates = control_dataset["forecast_date"].unique()


Great Now that we have a basic data set, we'll fill in the pump data with values for the next 7 days. These need to be reflective of the previous data, too, so we'll use the control data to fill in some of the data.

These are the columns for pumping:
'pump_cfs_m1', 'pump_cfs_m2', 'pump_cfs_m3'

First, we need transformed values to enter into the array. 

In [7]:
transformations = pd.read_csv("/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/data/mu_sigma.csv", index_col=0)

# get the index callled "pump_cfs"
x = transformations.loc[transformations.index == "pump_cfs", :].values
# calculate z score using the mean (first value) and the std (second value)
transformed_220 = (220 - x[0][0]) / x[0][1]
transformed_440 = (440 - x[0][0]) / x[0][1]

print(transformed_220, transformed_440)

0.22381065604391792 1.3177978994971216


And now, using the functions for 'static' and 'pulsed' operational regimes, we'll create two datasets for forecasting.

In [10]:
# create a dataframe for static pump of 220, using the control data to manipulate the m1, m2, m3 columns.
static_pump = pd.concat([static_regime_tiny(data = pumping_dataset, control = control_dataset, one_date = d, flow = transformed_220) for d in unique_dates])

# do the same for pulsing pumped, where there is a separate weekday/weekend flow
pulsed_pump = pd.concat([pulsed_regime_tiny(data = pumping_dataset, control = control_dataset, one_date = d, weekday_flow = transformed_440, weekend_flow = transformed_220) for d in unique_dates])

# Roll out forecast

Here, we're just iteratively modeling, for each of the 4 models for the next 7 days. First, we need to remove any columns that aren't in the model proper, and we need to make sure the order of the columns is the same as the training set. To do this, we'll just select the feature names in the proper order.

In [11]:
control_dataset.columns

Index(['mean_1m_temp_degC', 'mean_0_5m_temp_degC', 'mean_1m_temp_degC_m1',
       'mean_0_5m_temp_degC_m1', 'total_solar_radiation',
       'total_solar_radiation_m1', 'mean_air_temp', 'min_air_temp',
       'max_air_temp', 'mean_air_temp_m1', 'min_air_temp_m1',
       'max_air_temp_m1', 'mean_rel_hum_m1', 'pump_cfs_m1', 'mean_wind',
       'max_wind', 'mean_wind_m1', 'max_wind_m1', 'nf_cfs_m1',
       'chipmunk_cfs_m1', 'forecast_date'],
      dtype='object')

In [14]:
# get the feature names from the operational training file
upstream_names = pd.read_csv(os.path.join( "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/operational_model/data/training_1.csv"))
# reduce to fewer variables
drop_cols = ['total_solar_radiation_m2', 'mean_rel_hum_m2', 
       'pump_cfs_m2', 'pump_cfs_m3', 'nf_cfs_m2', 'nf_cfs_m3', 'nf_cfs_m4',
       'chipmunk_cfs_m2', 'chipmunk_cfs_m3', 'chipmunk_cfs_m4']

feature_names = upstream_names.drop(columns = drop_cols).columns

# remove date, and two target columns (first 3 columns)
feature_names = feature_names[3:]
feature_names = list(feature_names) + ["forecast_date"]

# and get those features in order for each dataset! (but we also need the forecast date!)
control_for_model = control_dataset[feature_names].copy()
static_for_model = static_pump[feature_names].copy()
pulsing_for_model = pulsed_pump[feature_names].copy()


And now rollout the forecast per unique valid date.

In [15]:
control_forecasts = (pd.concat([rollout_forecast(data = control_for_model, 
                                                 m1 = model_1, m2 = model_2, m3 = model_3, m4 = model_4, 
                                                 fore_date = d) for d in unique_dates]))

2025-05-16 14:53:25.414248: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


In [16]:
# we need to back transform the data for mean 1m and mean 0.5m
control_forecasts["mean_1m_temp_degC"] = control_forecasts["mean_1m_temp_degC"] * transformations.loc["mean_1m_temp_degC", "sd"] + transformations.loc["mean_1m_temp_degC", "mean"]
control_forecasts["mean_0_5m_temp_degC"] = control_forecasts["mean_0_5m_temp_degC"] * transformations.loc["mean_0_5m_temp_degC", "sd"] + transformations.loc["mean_0_5m_temp_degC", "mean"]

export_features = ["forecast_date", "valid_date", "model", "mean_1m_temp_degC", "mean_0_5m_temp_degC"]
control_forecasts[export_features]
# save the files to csv
control_forecasts.to_csv(os.path.join(dump_dir, "forecasted_temp_control_collated.csv"), index=False)

In [17]:
static_forecast = (pd.concat([rollout_forecast(data = static_for_model, 
                                                 m1 = model_1, m2 = model_2, m3 = model_3, m4 = model_4, 
                                                 fore_date = d) for d in unique_dates]))

In [18]:
# we need to back transform the data for mean 1m and mean 0.5m
static_forecast["mean_1m_temp_degC"] = static_forecast["mean_1m_temp_degC"] * transformations.loc["mean_1m_temp_degC", "sd"] + transformations.loc["mean_1m_temp_degC", "mean"]
static_forecast["mean_0_5m_temp_degC"] = static_forecast["mean_0_5m_temp_degC"] * transformations.loc["mean_0_5m_temp_degC", "sd"] + transformations.loc["mean_0_5m_temp_degC", "mean"]

# only grab the features we care about
static_forecast[export_features]

# save the files to csv
static_forecast.to_csv(os.path.join(dump_dir, "forecasted_temp_static_collated.csv"), index=False)

In [ ]:
pulsing_forecast = (pd.concat([rollout_forecast(data = pulsing_for_model, 
                                                 m1 = model_1, m2 = model_2, m3 = model_3, m4 = model_4, 
                                                 fore_date = d) for d in unique_dates]))

In [ ]:
# we need to back transform the data for mean 1m and mean 0.5m
pulsing_forecast["mean_1m_temp_degC"] = pulsing_forecast["mean_1m_temp_degC"] * transformations.loc["mean_1m_temp_degC", "sd"] + transformations.loc["mean_1m_temp_degC", "mean"]
pulsing_forecast["mean_0_5m_temp_degC"] = pulsing_forecast["mean_0_5m_temp_degC"] * transformations.loc["mean_0_5m_temp_degC", "sd"] + transformations.loc["mean_0_5m_temp_degC", "mean"]

# only grab the features we care about
pulsing_forecast[export_features]

# save the files to csv
pulsing_forecast.to_csv(os.path.join(dump_dir, "forecasted_temp_pulsing_collated.csv"), index=False)